# Aula 1 - Self-Attention

In [ ]:
import numpy as np
import math

# L = linhas.
L, d_k, d_v = 4,8,8

q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

In [3]:
print("Q\n",q,"\nK\n",k,"\nV\n",v)

Q
 [[ 0.42887222  0.74658826 -1.1421766  -0.26388489 -0.45459405  0.58156859
  -0.18005944 -0.172529  ]
 [ 1.16300157 -1.73641256 -0.22729429 -0.61397911 -1.54180775 -0.36408526
  -0.69725375  0.24922203]
 [-0.34976517 -0.36662488 -0.21640152 -0.04036618 -0.43403036 -0.93798797
   0.53623208  1.77276343]
 [-0.73683292 -1.20217703 -1.22920999 -0.05157512 -0.52157174 -1.55241842
   1.63690499 -0.61774941]] 
K
 [[-0.8318143   0.58309752 -0.03473482 -0.75222166 -0.44294655 -0.38208953
   0.53070577 -0.80894216]
 [-1.31226492  0.13880523  1.98615629 -0.72376571 -1.34321687 -1.19288675
   0.63365064  1.52687571]
 [ 1.36900118  0.39439524  0.6904862  -0.23942069  0.4797272  -1.65888696
   0.16253542 -0.33787671]
 [ 3.05693821  0.11547168 -1.20351437  1.52832332 -0.81341487  2.25198089
  -1.32581529  0.40576225]] 
V
 [[ 0.14429581 -1.0427667   0.67152451  0.18302065  2.39492649 -0.9551012
  -0.89128249  1.75327675]
 [-0.10129334  0.63285094 -0.26471826  1.02970102 -0.91510679  1.79765687
  -1.

## Self Attention

$\text{Self Attention} = \text{Softmax}(\frac{Q \cdot K^T}{sqrt(d_k)} + M)V$

In [4]:
np.matmul(q, k.T)

array([[ 0.33992194, -2.99736618, -0.99771044,  4.21674491],
       [-1.25974698,  0.66975619,  0.56416783,  4.14968168],
       [-0.4837914 ,  4.75598706,  0.07279928, -2.66370399],
       [ 2.18604372,  1.04241933,  0.48059028, -6.48337714]])

In [5]:
# Por que precisamos de sqrt(d_k) no denominador
q.var(), k.var(), np.matmul(q, k.T).var()

(np.float64(0.6983961766654991),
 np.float64(1.2919991744921493),
 np.float64(7.795491255483972))

In [7]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), scaled.var()

(np.float64(0.6983961766654991),
 np.float64(1.2919991744921493),
 np.float64(0.9744364069354966))

Veja que a variância diminui drasticamente.

## Masking

- Assegura que não olhe em palavras futuras quando tentar gerar o contexto atual (isso seria trapaça) 
- Não necessário em encoders, mas obrigatório em decoders

In [9]:
mask = np.tril(np.ones((L,L))) # Matriz triangular
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

Com o exemplo "My name is Ajay", nessa simulação de máscara, cada palavra só pode olhar para ela mesma e para trás.

In [11]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

Os valores com -inf não terão contexto nenhum (por causa da Softmax, que esmaga valores muito negativos)

In [12]:
scaled + mask

array([[ 0.12018056,        -inf,        -inf,        -inf],
       [-0.44538782,  0.23679457,        -inf,        -inf],
       [-0.17104609,  1.68149535,  0.02573843,        -inf],
       [ 0.77288317,  0.36855089,  0.16991432, -2.29221997]])

## Softmax

$$\text{Softmax} = \frac{e^{x_i}}{\sum_j e^x_j}$$

In [ ]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=-1)).T

# A softmax será aplicada para cada linha da matriz final.

attention = softmax(scaled + mask)
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.33577439, 0.66422561, 0.        , 0.        ],
       [0.11636723, 0.74195781, 0.14167497, 0.        ],
       [0.44223209, 0.29515557, 0.24198264, 0.0206297 ]])

Isso significa:

- My - Foca apenas em "My"
- Name - Foca em My,Name
- is - Foca em My, Name, is
- Ajay - Foca em todas

In [16]:
new_v = np.matmul(attention, v)
new_v

array([[ 0.14429581, -1.0427667 ,  0.67152451,  0.18302065,  2.39492649,
        -0.9551012 , -0.89128249,  1.75327675],
       [-0.01883079,  0.07022145,  0.04964809,  0.74540743,  0.19631761,
         0.87335121, -1.08935053,  1.47109624],
       [-0.0383209 ,  0.21936916, -0.22505917,  0.65561853, -0.46174139,
         1.09681476, -0.96755626,  1.12504376],
       [ 0.05854839, -0.46844205,  0.06537077,  0.14782814,  0.69706077,
        -0.09916435, -0.75704287,  1.05573576]])

Essa nova matriz encapsula melhor o contexto de uma palavra.

In [17]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.shape[-1]
    scaled = np.matmul(q, k.T) / math.sqrt(d_k)
    if mask is not None:
        scaled = scaled + mask
    attention = softmax(scaled)
    out = np.matmul(attention, v)
    return out, attention